# PTC Receiver Analysis
![alt text](ptc_thermal_model_radial.png)

In [ ]:
import numpy as np
from CoolProp.CoolProp import PropsSI
from enn554.math import cosd
sigma = 5.670e-8  
import matplotlib.pyplot as plt
from scipy.optimize import minimize, Bounds

Parameters

In [ ]:
T_inf, T_sky = 22 + 273.15, 22 - 8 + 273.15
U = 5.0 # m/s wind speed
mdot_htf = 10 #kg/s
eta_0 = 0.81

tau_env = 0.97
eps_env = 0.86
alpha_env = 0.02

alpha_abs = 0.955
eps_abs = 0.14 # simplification here, likely a function of Ta

HTF_name = 'INCOMP::TVP1' 
Dia,Doa = 6.6e-2, 7.0e-2
Die, Doe = 10.9e-3, 11.5e-3 
W = 4.8235 # m

theta,DNI = 10, 950
I_bn = DNI*W

HTF side Nusselt number

In [ ]:
def nusselt_htf(T,pressure=1e7):
    T_f,T_a,T_e = T

    k = PropsSI('conductivity','T',T_f,'P',pressure,HTF_name)
    Pr = PropsSI('Prandtl','T',T_f,'P',pressure,HTF_name)
    Prw = PropsSI('Prandtl','T',T_a,'P',pressure,HTF_name)
    mu = PropsSI('viscosity','T',T_f,'P',pressure,HTF_name)
    ReD = 4*mdot_htf/(np.pi*Dia*mu)

    if ReD > 5e6:
        print('Warning: Reynolds number if outside the range of validity.')

    if ReD < 2300: # laminar
        Nu = 4.36
    else:
        f2 = (1.82*np.log10(ReD)-1.64)**(-2.0)
        Nu = f2/8.0 * (ReD-100)*Pr / (1+12.7*np.sqrt(f2/8.0) * (Pr**(2.0/3.0) - 1)) *(Pr/Prw)**0.11

    return np.pi*Nu*k*(T_a - T_f)


Envelop Nusselt

In [ ]:
def nusselt_envelope(Te,air_pressure=101.3e3):
    T_film = 0.5*(Te+T_inf)
    D = Doe
    mu = PropsSI('viscosity','T',T_film,'P',air_pressure,'Air')
    rho = PropsSI('D','T',T_film,'P',air_pressure,'Air')
    
    if U<0.1: # No wind, Natural Convec
        Pr_film = PropsSI('Prandtl','T',T_film,'P',air_pressure,'Air')
        β = PropsSI('isobaric_expansion_coefficient','T',T_film,'P',air_pressure,'Air')
        k = PropsSI('conductivity','T',T_film,'P',air_pressure,'Air')
        cp = PropsSI('C','T',T_film,'P',air_pressure,'Air')
        α = k/rho/cp
        v = mu/rho            
        
        RaD = 9.81*β*D**3*(Te-T_inf)/(α*v)
        den = (1+ (0.559/Pr_film)**(9/16))**(16/9)
        Nu = (0.6 + 0.387*(RaD/den)**(1/6) )**2
    else: # Forced convection
        Pr_e = PropsSI('Prandtl','T',Te,'P',air_pressure,'Air')
        Pr_ambient = PropsSI('Prandtl','T',T_inf,'P',air_pressure,'Air')
        ReD = rho * U * D / mu

        if (Pr_ambient < 0.7) or (Pr_ambient>500):
            print('Warning: Prandtl number if outside the range of validity in the Nu computation.')
        if (ReD < 1.0) or (ReD > 1e6):
            print('Warning: Reynolds number if outside the range of validity in the Nu computation.')

        if ReD <= 40:
            C,m = 0.75,0.4
        elif ReD <= 1000:
            C,m = 0.51,0.5
        elif ReD<= 200e3:
            C,m = 0.26,0.6
        else:
            C,m = 0.076,0.7

        if Pr_ambient<=10:
            n=0.37
        else:
            n=0.36
        
        Nu = C*ReD**m * Pr_ambient**n * (Pr_ambient/Pr_e)**(1/4.0)
    
    return Nu

Radiation helper

In [ ]:
def Q_ae_rad(T):
    T_f,T_a,T_e = T
    num = sigma*np.pi*Doa*(T_a**4-T_e**4)
    den = 1/eps_abs + (1-eps_env)/eps_env * (Doa/Die)
    return num/den

Compute residuals for particular temperatures

In [ ]:
IAM = cosd(theta) + 0.000884*theta - 0.00005369*theta**2
def residuals(T):
    T_f,T_a,T_e = T
    Q_a_abs = eta_0 * IAM * tau_env * alpha_abs * I_bn
    Q_e_abs = eta_0 * IAM * alpha_env * I_bn

    Q_af_conv = np.pi*nusselt_htf(T) * PropsSI('conductivity','T',T_f,'P',1e7,HTF_name)*(T_a-T_f)
    k_film = PropsSI('conductivity','T',0.5*(T_inf+T_e),'P',101.3e3,HTF_name)
    Q_esa_conv = np.pi*nusselt_envelope(T_e)*k_film*(T_e-T_inf)
    Q_es_rad = sigma*Doe*np.pi*eps_env * (T_e**4 - T_sky**4)
    Q_ae_rad_val = Q_ae_rad(T)

    res_abs = Q_a_abs - Q_af_conv - Q_ae_rad_val
    res_env = Q_e_abs + Q_ae_rad_val - Q_es_rad - Q_esa_conv

    return res_abs**2 + res_env**2

In [ ]:
T_htf = 373.15
bnd = Bounds((286,286),(670,670),keep_feasible=True)
obj = lambda x: residuals([T_htf,x[0],x[1]])
sol = minimize(obj,[T_htf+1.0,T_inf],bounds=bnd)

In class, I was missing the -273.15 in line 12 below to convert T_htf to K. If I had to do it over again, I would have converted T_htf to K in the first line after "for".

In [ ]:
heat_gains = []
heat_losses = []
efficiency = []
temps = [100,150,200,250,300,350,390] # C

for T_htf in temps:
    T0 = [T_htf+273.15+10,T_inf]
    obj = lambda x: residuals([T_htf+273.15,x[0],x[1]])
    sol = minimize(obj,T0,bounds=bnd)
    Ts = [T_htf+273.15, sol.x[0],sol.x[1]]

    g = np.pi*nusselt_htf(Ts) * PropsSI('conductivity','T',T_htf+273.15,'P',1e7,HTF_name)*(Ts[1]-T_htf - 273.15)

    heat_gains.append(g)
    efficiency.append(g/DNI/W) 


In [ ]:
fig,ax = plt.subplots()
ax.plot(temps,efficiency,color='blue')

ax2 = ax.twinx()
ax2.plot(temps,DNI*W - np.array(heat_gains),color='orange')


ax.set_xlabel('HTF Temperature')
ax2.set_ylabel('Heat Gain (W/m)', color='orange')
ax2.tick_params(axis='y', colors='orange')
ax2.spines['right'].set_color('orange')

ax.set_ylabel('Efficiency', color='blue')
ax.tick_params(axis='y', colors='blue')
ax.spines['left'].set_color('blue')
